# Notebook 01 — Dataset Audit and Master Manifest
Binary classification: tongue images for diabetes screening.
This notebook scans, validates, and audits the dataset. No model training.


In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'imagehash', '-q'])
print('imagehash ready.')


imagehash ready.


## Configuration


In [2]:
import os, hashlib, re, itertools
from pathlib import Path
import pandas as pd
from PIL import Image
import imagehash

DATASET_ROOT = Path(r'D:\DIABETES\Type 2 Diabetes Mellitus Tongue Dataset')
OUTPUT_DIR   = Path(r'D:\DIABETES\diabetes_pipeline_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_SOURCE = 'type2_diabetes_mellitus_tongue_dataset'
VALID_SPLITS   = {'train', 'valid', 'test'}
VALID_EXTS     = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

LABEL_MAP = {
    'diabetes':    ('diabetes',    1),
    'nondiabetes': ('non_diabetes', 0),
}

AUG_TOKENS = [
    'aug', 'augmented', 'rotate', 'rotated', 'rot', 'flip', 'flipped',
    'hflip', 'vflip', 'mirror', 'crop', 'cropped', 'zoom', 'brightness',
    'bright', 'contrast', 'noise', 'blur', 'sharpen', 'original', 'copy'
]
AUG_PATTERN = re.compile(
    r'[_\-]?(' + '|'.join(AUG_TOKENS) + r')[_\-]?\d*', re.IGNORECASE
)

NEAR_DUP_STRONG = 4
NEAR_DUP_POSSIBLE = 8

print('Configuration loaded.')
print(f'Dataset root : {DATASET_ROOT}')
print(f'Output dir   : {OUTPUT_DIR}')


Configuration loaded.
Dataset root : D:\DIABETES\Type 2 Diabetes Mellitus Tongue Dataset
Output dir   : D:\DIABETES\diabetes_pipeline_outputs


## Helper Functions


In [3]:
def md5_of_file(path: Path) -> str:
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()


def derive_family_id(stem: str) -> str:
    """Strip augmentation tokens from a filename stem to get a base family id."""
    base = AUG_PATTERN.sub('', stem)
    base = re.sub(r'\d+$', '', base)   # trailing digits
    base = re.sub(r'[_\-]+$', '', base) # trailing separators
    return base.strip().lower() or stem.lower()


def validate_image(path: Path):
    """Return (readable, width, height, mode). On failure return (False, None, None, None)."""
    try:
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:
            return True, img.width, img.height, img.mode
    except Exception:
        return False, None, None, None


def compute_phash(path: Path):
    try:
        with Image.open(path) as img:
            return str(imagehash.phash(img))
    except Exception:
        return None

print('Helpers defined.')


Helpers defined.


## Scan Dataset


In [4]:
records = []
image_id = 0

for split in VALID_SPLITS:
    split_dir = DATASET_ROOT / split
    if not split_dir.exists():
        print(f'WARNING: split folder not found: {split_dir}')
        continue
    for class_dir in split_dir.iterdir():
        if not class_dir.is_dir():
            continue
        folder_label = class_dir.name.lower()
        if folder_label not in LABEL_MAP:
            print(f'WARNING: unexpected class folder: {class_dir}')
            continue
        final_label, label_binary = LABEL_MAP[folder_label]
        for img_path in class_dir.rglob('*'):
            if img_path.suffix.lower() not in VALID_EXTS:
                continue
            image_id += 1
            stem = img_path.stem
            records.append({
                'image_id':       image_id,
                'file_path':      str(img_path),
                'dataset_source': DATASET_SOURCE,
                'original_split': split,
                'folder_label':   folder_label,
                'final_label':    final_label,
                'label_binary':   label_binary,
                'filename':       img_path.name,
                'stem':           stem,
                'extension':      img_path.suffix.lower(),
                'file_size_bytes': img_path.stat().st_size,
                '_path_obj':      img_path,
            })

print(f'Total image files found: {len(records)}')


Total image files found: 2750


## Validate Images and Compute Hashes


In [5]:
from tqdm import tqdm

for rec in tqdm(records, desc='Validating & hashing'):
    p = rec['_path_obj']
    readable, w, h, mode = validate_image(p)
    rec['readable']     = readable
    rec['width']        = w
    rec['height']       = h
    rec['image_mode']   = mode
    rec['exact_hash_md5']       = md5_of_file(p)
    rec['perceptual_hash_phash'] = compute_phash(p) if readable else None
    rec['audit_status'] = 'excluded_unreadable' if not readable else 'ok'
    rec['audit_notes']  = 'unreadable/corrupted' if not readable else ''

df = pd.DataFrame(records).drop(columns=['_path_obj'])
print(f'Validation complete. Unreadable: {(~df["readable"]).sum()}')


Validating & hashing: 100%|██████████| 2750/2750 [00:16<00:00, 162.31it/s]

Validation complete. Unreadable: 0


## Detect Exact Duplicates


In [6]:
md5_counts = df['exact_hash_md5'].value_counts()
dup_md5s   = set(md5_counts[md5_counts > 1].index)

dup_group_map = {}
group_id = 0
for md5, grp in df[df['exact_hash_md5'].isin(dup_md5s)].groupby('exact_hash_md5'):
    group_id += 1
    for iid in grp['image_id']:
        dup_group_map[iid] = f'exact_dup_{group_id:04d}'

df['exact_duplicate_group_id'] = df['image_id'].map(dup_group_map)
print(f'Exact duplicate groups: {group_id}')
print(f'Images in exact duplicate groups: {df["exact_duplicate_group_id"].notna().sum()}')


Exact duplicate groups: 0
Images in exact duplicate groups: 0


## Detect Filename Augmentation Families


In [7]:
df['_family_base'] = df['stem'].apply(derive_family_id)

# Only create a family group if multiple images share the same base
family_counts = df['_family_base'].value_counts()
multi_families = set(family_counts[family_counts > 1].index)

family_id_counter = 0
family_id_map = {}
for base in multi_families:
    family_id_counter += 1
    family_id_map[base] = f'family_{family_id_counter:04d}'

df['filename_family_id'] = df['_family_base'].map(family_id_map)
df.drop(columns=['_family_base'], inplace=True)

print(f'Filename family groups: {family_id_counter}')
print(f'Images in family groups: {df["filename_family_id"].notna().sum()}')


Filename family groups: 900
Images in family groups: 2250


## Detect Near-Duplicate Candidates (Perceptual Hash)


In [8]:
readable_df = df[df['perceptual_hash_phash'].notna()].copy()
readable_df['_phash_obj'] = readable_df['perceptual_hash_phash'].apply(imagehash.hex_to_hash)

near_dup_pairs = []
rows = list(readable_df[['image_id', '_phash_obj']].itertuples(index=False))

for (id_a, ph_a), (id_b, ph_b) in itertools.combinations(rows, 2):
    dist = ph_a - ph_b
    if dist <= NEAR_DUP_POSSIBLE:
        near_dup_pairs.append({
            'image_id_a': id_a,
            'image_id_b': id_b,
            'hamming_distance': dist,
            'strength': 'strong' if dist <= NEAR_DUP_STRONG else 'possible'
        })

nd_pairs_df = pd.DataFrame(near_dup_pairs)
print(f'Near-duplicate pairs found: {len(nd_pairs_df)}')
if not nd_pairs_df.empty:
    print(nd_pairs_df['strength'].value_counts().to_string())

# Assign near_duplicate_group_id using union-find on strong pairs
parent = {iid: iid for iid in df['image_id']}

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(x, y):
    parent[find(x)] = find(y)

if not nd_pairs_df.empty:
    for _, row in nd_pairs_df[nd_pairs_df['strength'] == 'strong'].iterrows():
        union(int(row['image_id_a']), int(row['image_id_b']))

# Map roots to group ids
root_counts = {}
for iid in df['image_id']:
    r = find(iid)
    root_counts.setdefault(r, [])
    root_counts[r].append(iid)

nd_group_map = {}
nd_counter = 0
for root, members in root_counts.items():
    if len(members) > 1:
        nd_counter += 1
        for iid in members:
            nd_group_map[iid] = f'near_dup_{nd_counter:04d}'

df['near_duplicate_group_id'] = df['image_id'].map(nd_group_map)
print(f'Near-duplicate groups (strong): {nd_counter}')


Near-duplicate pairs found: 3757
strength
possible    2812
strong       945
Near-duplicate groups (strong): 323


## Assign Effective Group ID


In [9]:
def assign_effective_group(row):
    if pd.notna(row['filename_family_id']):
        return row['filename_family_id']
    if pd.notna(row['exact_duplicate_group_id']):
        return row['exact_duplicate_group_id']
    if pd.notna(row['near_duplicate_group_id']):
        return row['near_duplicate_group_id']
    return f"img_{row['image_id']}"

df['effective_group_id'] = df.apply(assign_effective_group, axis=1)
print('Effective group IDs assigned.')


Effective group IDs assigned.


## Finalise Manifest Column Order


In [10]:
MANIFEST_COLS = [
    'image_id', 'file_path', 'dataset_source', 'original_split',
    'folder_label', 'final_label', 'label_binary',
    'filename', 'stem', 'extension',
    'width', 'height', 'image_mode', 'file_size_bytes', 'readable',
    'exact_hash_md5', 'perceptual_hash_phash',
    'filename_family_id', 'exact_duplicate_group_id',
    'near_duplicate_group_id', 'effective_group_id',
    'audit_status', 'audit_notes'
]

manifest = df[MANIFEST_COLS].copy()
print(f'Manifest shape: {manifest.shape}')


Manifest shape: (2750, 23)


## Save All Outputs


In [11]:
# 1. Master manifest
manifest.to_csv(OUTPUT_DIR / '01_master_manifest.csv', index=False)
print(f'Saved: 01_master_manifest.csv')

# 2. Dataset summary
summary = {
    'total_images': len(manifest),
    'unreadable': int((~manifest['readable']).sum()),
    'exact_duplicate_groups': int(manifest['exact_duplicate_group_id'].notna().sum() > 0) * manifest['exact_duplicate_group_id'].nunique(),
    'filename_family_groups': int(manifest['filename_family_id'].nunique()),
    'near_duplicate_strong_groups': nd_counter,
    'near_duplicate_pairs_total': len(nd_pairs_df),
}
by_split  = manifest['original_split'].value_counts().to_dict()
by_class  = manifest['final_label'].value_counts().to_dict()
by_split_class = manifest.groupby(['original_split','final_label']).size().to_dict()

summary_rows = [{'metric': k, 'value': v} for k, v in summary.items()]
for k, v in by_split.items():  summary_rows.append({'metric': f'split_{k}', 'value': v})
for k, v in by_class.items():  summary_rows.append({'metric': f'class_{k}', 'value': v})
for (sp, cl), v in by_split_class.items(): summary_rows.append({'metric': f'{sp}_{cl}', 'value': v})

# Image size stats
readable_m = manifest[manifest['readable']]
for stat, val in readable_m['width'].describe().items():
    summary_rows.append({'metric': f'width_{stat}', 'value': round(val, 2)})
for stat, val in readable_m['height'].describe().items():
    summary_rows.append({'metric': f'height_{stat}', 'value': round(val, 2)})

pd.DataFrame(summary_rows).to_csv(OUTPUT_DIR / '01_dataset_summary.csv', index=False)
print('Saved: 01_dataset_summary.csv')

# 3. Exact duplicates
exact_dup_df = manifest[manifest['exact_duplicate_group_id'].notna()][[
    'image_id','file_path','original_split','final_label','exact_hash_md5','exact_duplicate_group_id'
]]
exact_dup_df.to_csv(OUTPUT_DIR / '01_exact_duplicates.csv', index=False)
print(f'Saved: 01_exact_duplicates.csv ({len(exact_dup_df)} rows)')

# 4. Near duplicate candidates
if not nd_pairs_df.empty:
    nd_pairs_df.to_csv(OUTPUT_DIR / '01_near_duplicate_candidates.csv', index=False)
else:
    pd.DataFrame(columns=['image_id_a','image_id_b','hamming_distance','strength']).to_csv(
        OUTPUT_DIR / '01_near_duplicate_candidates.csv', index=False)
print(f'Saved: 01_near_duplicate_candidates.csv ({len(nd_pairs_df)} pairs)')

# 5. Family group summary
family_summary = manifest[manifest['filename_family_id'].notna()].groupby('filename_family_id').agg(
    count=('image_id','count'),
    splits=('original_split', lambda x: ','.join(sorted(x.unique()))),
    labels=('final_label', lambda x: ','.join(sorted(x.unique())))
).reset_index()
family_summary.to_csv(OUTPUT_DIR / '01_family_group_summary.csv', index=False)
print(f'Saved: 01_family_group_summary.csv ({len(family_summary)} groups)')

# 6. Audit log
audit_lines = [
    'DIABETES PIPELINE — NOTEBOOK 01 AUDIT LOG',
    '=' * 50,
    f'Dataset source : {DATASET_SOURCE}',
    f'Dataset root   : {DATASET_ROOT}',
    f'Total images   : {len(manifest)}',
    f'Unreadable     : {int((~manifest["readable"]).sum())}',
    '',
    'By split:',
    *[f'  {k}: {v}' for k,v in by_split.items()],
    '',
    'By class:',
    *[f'  {k}: {v}' for k,v in by_class.items()],
    '',
    'By split + class:',
    *[f'  {sp}/{cl}: {v}' for (sp,cl),v in by_split_class.items()],
    '',
    'Duplicate / family analysis:',
    f'  Exact duplicate groups : {summary["exact_duplicate_groups"]}',
    f'  Filename family groups : {summary["filename_family_groups"]}',
    f'  Near-dup strong groups : {nd_counter}',
    f'  Near-dup pairs total   : {len(nd_pairs_df)}',
    '',
    'LIMITATION: Patient identifiers are unavailable. Filename-family',
    'grouping, exact hashing, and perceptual hashing reduce leakage risk',
    'from augmented or near-duplicate images, but they do not guarantee',
    'perfect patient-level independence if the same patient appears under',
    'unrelated filenames.',
]
with open(OUTPUT_DIR / '01_audit_log.txt', 'w') as f:
    f.write('\n'.join(audit_lines))
print('Saved: 01_audit_log.txt')


Saved: 01_master_manifest.csv
Saved: 01_dataset_summary.csv
Saved: 01_exact_duplicates.csv (0 rows)
Saved: 01_near_duplicate_candidates.csv (3757 pairs)
Saved: 01_family_group_summary.csv (900 groups)
Saved: 01_audit_log.txt


## Summary Report


In [12]:
print('=' * 50)
print('NOTEBOOK 01 — FINAL SUMMARY')
print('=' * 50)
print(f'Total images          : {len(manifest)}')
print(f'Readable              : {manifest["readable"].sum()}')
print(f'Unreadable/excluded   : {(~manifest["readable"]).sum()}')
print()
print('By original split:')
print(manifest['original_split'].value_counts().to_string())
print()
print('By class:')
print(manifest['final_label'].value_counts().to_string())
print()
print('By split + class:')
print(manifest.groupby(['original_split','final_label']).size().to_string())
print()
print('Duplicate / family analysis:')
print(f'  Exact duplicate groups : {summary["exact_duplicate_groups"]}')
print(f'  Filename family groups : {summary["filename_family_groups"]}')
print(f'  Near-dup strong groups : {nd_counter}')
print(f'  Near-dup pairs total   : {len(nd_pairs_df)}')
print()
print('Image size (readable images):')
print(f'  Width  — min {readable_m["width"].min()}, max {readable_m["width"].max()}, mean {readable_m["width"].mean():.0f}')
print(f'  Height — min {readable_m["height"].min()}, max {readable_m["height"].max()}, mean {readable_m["height"].mean():.0f}')
print()
print('Outputs saved to:', OUTPUT_DIR)
print('=' * 50)

print()
print('LIMITATION: Patient identifiers are unavailable. Filename-family grouping,')
print('exact hashing, and perceptual hashing reduce leakage risk from augmented or')
print('near-duplicate images, but they do not guarantee perfect patient-level')
print('independence if the same patient appears under unrelated filenames.')


NOTEBOOK 01 — FINAL SUMMARY
Total images          : 2750
Readable              : 2750
Unreadable/excluded   : 0

By original split:
original_split
train    2100
valid     600
test       50

By class:
final_label
diabetes        1375
non_diabetes    1375

By split + class:
original_split  final_label 
test            diabetes          25
                non_diabetes      25
train           diabetes        1050
                non_diabetes    1050
valid           diabetes         300
                non_diabetes     300

Duplicate / family analysis:
  Exact duplicate groups : 0
  Filename family groups : 900
  Near-dup strong groups : 323
  Near-dup pairs total   : 3757

Image size (readable images):
  Width  — min 214, max 2600, mean 326
  Height — min 241, max 3475, mean 327

Outputs saved to: D:\DIABETES\diabetes_pipeline_outputs

LIMITATION: Patient identifiers are unavailable. Filename-family grouping,
exact hashing, and perceptual hashing reduce leakage risk from augmented or
near-